# Week 10 · 部署、缓存、队列与系统设计

请求经过 DNS、HTTPS 反向代理、应用服务、检索/数据库、模型，再返回客户端。DNS 只映射名称，不负责运行应用；反向代理终止 TLS、转发请求。进程存活检查(liveness)与依赖就绪检查(readiness)不同。公开部署之前必须考虑认证、配额、上传限制、备份和日志。

缓存适合复用结果，需要 key、TTL 和失效策略；用户身份/权限影响结果时必须进入 key，否则可能泄露私人数据。队列把耗时工作移到后台，但带来重复投递、失败恢复和可观测性问题。内存 asyncio.Queue 只演示调度，进程重启会丢工作；可靠任务需要持久化队列。

同步模型简单但等待会阻塞请求，异步改善并发等待但增加取消与状态管理。先测瓶颈再扩容，检索、模型、数据库各有不同约束。下面只演示本地机制，不执行云部署，不宣称已获得公网 Demo。

## 学习方式 / How to study
先预测代码结果，再逐行运行。改变一个输入、解释变化，最后不看参考实现重写关键函数。阅读不是掌握的证据；能独立实现、测试、解释失败才是。

In [ ]:
import asyncio,time
from dataclasses import dataclass,field
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pathlib import Path

@dataclass
class TTLCache:
    ttl:float
    values:dict=field(default_factory=dict)
    def get(self,key):
        value=self.values.get(key)
        if not value:return None
        if time.monotonic()>=value[0]:
            self.values.pop(key,None);return None
        return value[1]
    def put(self,key,value):self.values[key]=(time.monotonic()+self.ttl,value)

cache=TTLCache(ttl=.05)
cache.put(("user-a","document-1","query"),"private answer")
assert cache.get(("user-b","document-1","query")) is None
await asyncio.sleep(.06)
assert cache.get(("user-a","document-1","query")) is None

async def pipeline():
    queue=asyncio.Queue(maxsize=2);completed=[]
    async def worker():
        while True:
            item=await queue.get()
            try:
                if item is None:return
                await asyncio.sleep(.01)
                completed.append({"id":item,"state":"indexed"})
            finally:queue.task_done()
    task=asyncio.create_task(worker())
    for identifier in range(5):await queue.put(identifier)
    await queue.put(None);await queue.join();await task
    return completed
print(await pipeline())
app=FastAPI()
@app.get("/health/live")
def live():return {"status":"alive"}
@app.get("/health/ready")
def ready():return {"status":"ready","storage":"demo-memory"}
with TestClient(app) as client:assert client.get("/health/live").status_code==200
diagram="""flowchart LR
 Browser --> Proxy[HTTPS reverse proxy]
 Proxy --> API[FastAPI]
 API --> DB[(Database)]
 API --> Retrieval[Document index]
 API --> Model[Model service]
 API --> Queue[Background queue]
"""
Path("week10-architecture.mmd").write_text(diagram,encoding="utf-8")
print(diagram)

## 练习 / Exercises
解释缓存击穿、失效和用户隔离。设计后台文档解析状态：queued/running/succeeded/failed，并写故障恢复策略。

先在下面独立完成，再展开参考实现。

In [ ]:
# 在这里写你的实现；运行后检查边界。


## 参考实现与验收 / Reference and checks
参考实现是一个可行方案，不是唯一答案。不要在未完成练习前直接复制。

In [ ]:
states={"queued":{"running"},"running":{"succeeded","failed"},"failed":{"queued"},"succeeded":set()}
def transition(current,target):
    if target not in states[current]:raise ValueError("invalid transition")
    return target
assert transition("queued","running")=="running"
try:transition("succeeded","running")
except ValueError:print("已完成任务不会被意外重新执行")